# Workspaces and stages

A finished record is read-only: its key promises its contents. Two kinds of record relax this
in controlled ways:

- A **workspace record** (`mutable = True`) accepts writes after its build. A trainer logs
  losses and checkpoints over hours; a solver saves intermediate states. datamate served this
  with `Directory.__init__`.
- A **staged record** splits a long build into named stages, declared with `td.Stages`. It
  builds in its final directory, so readers can watch it and a failed build resumes after the
  last stage that finished.

This notebook walks through the status lifecycle of both.

In [ ]:
import tempfile
from pathlib import Path
from typing import Annotated

import numpy as np

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-workspaces-"))
td.set_root(root)

## A workspace record

`__init__` is the build, as for every record. Two parameter options matter for objects like a
trainer:

- `Annotated[T, td.field(hashed=False)]` keeps a setting in the spec but out of the key;
- `Annotated[T, td.transient]` keeps a parameter out of the spec altogether (a logger, a
  callback, a verbosity flag): `__init__` receives it, nothing stores it.

Names starting with `_` are plain Python attributes; every other attribute written on a record
is a stored field.

In [ ]:
log = []  # what each __init__ saw, for this notebook


class Trainer(td.Record):
    mutable = True

    def __init__(
        self,
        /,
        *,
        lr: float = 0.1,
        n_units: int = 4,
        seed: int = 0,
        device: Annotated[str, td.field(hashed=False)] = "cpu",  # in the spec, not in the key
        verbose: Annotated[bool, td.transient] = False,  # not in the spec
    ):
        log.append(("init", self.status, verbose))
        self.weights = np.random.default_rng(seed).normal(size=n_units)

    def train(self, n_steps: int = 20) -> float:
        if "summary" in self:  # finished earlier, maybe by another process
            return self.summary["final_loss"]
        weights = self.load("weights")
        for _ in range(n_steps):
            weights = weights * (1 - self.spec.lr)
            self.append("loss", [float((weights**2).sum())])
        self.weights = weights
        self.summary = {"final_loss": float(self.load("loss")[-1])}
        return self.summary["final_loss"]


Trainer.Spec()

## Lifecycle

| moment | status | writes |
|---|---|---|
| first call, `__init__` running | `running` (the call holds the record's lease) | go into the build |
| `__init__` returns | `done` | |
| `__init__` raises | `failed`, traceback stored; `BuildFailed` is raised | discarded |
| a later call with the same settings | stays `done`; `__init__` does not run | |
| method calls after the build | stays `done` | edits; each stamps `meta["modified"]` |

In [ ]:
print("before:", Trainer.locate().status)
trainer = Trainer(verbose=True)
print("during __init__:", log[-1])
print("after:", trainer.status, trainer.keys(), trainer.meta["modified"])

Methods keep writing after the build. Each edit takes the record's lease briefly, re-reads its
manifest, writes the field and stamps `modified`; the status stays `done`.

In [ ]:
final = trainer.train()
print(final, trainer.status, trainer.keys())
print("modified:", trainer.meta["modified"] is not None)

Calling the class again with the same settings reopens the record without running `__init__`,
so `train` can skip finished work, here or in another process. The unhashed `device` gives the
same record; another `lr` gives another one.

In [ ]:
n_inits = len(log)
again = Trainer(device="cuda")
print("__init__ ran again:", len(log) > n_inits, "| same record:", again == trainer, again.train() == final)
faster = Trainer(lr=0.3)
faster.train()
Trainer.info(sizes=False, varying_only=True)[["key", "status", "spec.lr"]]

## A failing build

An exception in `__init__` stores a `failed` record with the traceback and raises
`td.BuildFailed`. Later calls raise the stored failure again, without running `__init__`, until
`retry_failed=True` asks for another attempt or the record is deleted.

In [ ]:
attempts = []


class Flaky(td.Record):
    def __init__(self, /, *, threshold: float = 0.5):
        attempts.append(len(attempts))
        if len(attempts) == 1:
            raise RuntimeError("the GPU went away")
        self.ok = True


for _ in range(2):
    try:
        Flaky()
    except td.BuildFailed as exc:
        print("raised:", exc.error["type"], exc.error["message"], "| attempts:", len(attempts))

flaky = Flaky.get(retry_failed=True)
print("retried:", flaky.status, flaky.ok, "| attempts:", len(attempts))

## Holding a record

A class that should not be a record itself holds one. Concurrent handles edit the same
directory; each edit re-reads the manifest, so neither handle loses the other's fields.

In [ ]:
class Run(td.Record):
    mutable = True

    def __init__(self, /, *, lr: float = 0.1):
        pass  # an empty build: fields are written afterwards


class Solver:
    """An ordinary class that keeps its state in a workspace record."""

    def __init__(self, lr: float):
        self.record = Run(lr=lr)  # or Run.at(path, lr=lr) for a hand-named directory

    def step(self, losses):
        self.record.losses = np.asarray(losses)


solver = Solver(0.1)
other_handle = Run(lr=0.1)  # e.g. in another process
solver.step([1.0, 0.5])
other_handle.notes = {"comment": "converged"}
solver.record.append("steps", [1, 2])
print(sorted(Run(lr=0.1).keys()))
del solver.record.losses
print(sorted(other_handle.keys()), other_handle.status)

Writing a field again replaces it (the last write wins); writing an array over an appended
field replaces its rows and keeps it appendable.

In [ ]:
run = solver.record
run.steps = np.array([10, 20, 30])
run.append("steps", [40])
print(run.load("steps"))

## `in_place` builds

By default a build runs in a temporary directory that appears by rename when it finishes, so
nobody sees a half-built record. `in_place = True` builds in the final directory instead:

- readers see fields and `append`ed rows while the build runs (training logs);
- a retry after a failure keeps what the failed attempt wrote, so the build can resume from its
  last checkpoint.

In [ ]:
crash = {"at": 6}  # simulate a node failure at step 6 on the first attempt


class LongFit(td.Record):
    in_place = True

    def __init__(self, /, *, n_steps: int = 10):
        start = int(self.load("step")[-1]) + 1 if "step" in self else 0  # resume
        for step in range(start, n_steps):
            if step == crash["at"]:
                crash["at"] = None
                raise RuntimeError(f"node lost at step {step}")
            self.append("step", [step])
            self.append("loss", [1 / (step + 1)])
            if step == 2:
                reader = td.Record.at(self.path)  # another handle sees the rows written so far
                print("a reader during the build:", reader.status, reader.load("step").tolist())


try:
    LongFit()
except td.BuildFailed as exc:
    print("failed:", exc.error["message"], "| rows kept:", LongFit.locate().load("step").tolist())

fit = LongFit.get(retry_failed=True)
print("resumed:", fit.status, fit.load("step").tolist())

## Stages

`td.Stages` declares the stages of a build by name; each stage method is decorated with that
`Stages` instance and returns the record, so stages chain.

- `td.Stage(after=...)` runs only after the named stages, else `td.StageOrderError`;
- `repeat=True` allows many calls (epochs); other stages run once;
- `required=False` makes a stage optional: the build may skip it, and it may run on the
  finished record, adding fields but never rewriting others.

A staged record builds in place. A retried build skips the stages that finished, and as many
calls of a repeated stage as finished before, so stages hand results on through fields
(`self.x = ...`), not private attributes. Progress is stored in `meta["stages"]`.

In [ ]:
crash = {"at": 1}  # the first attempt fails in epoch 1
ran = []  # which stage calls ran, for this notebook


class Fit(td.Record):
    stages = td.Stages(
        prepare=td.Stage(),
        solve=td.Stage(after="prepare"),
        epoch=td.Stage(after="solve", repeat=True),
        preview=td.Stage(after="solve", required=False),
    )

    def __init__(self, /, *, n_points: int = 200, n_epochs: int = 3, seed: int = 0):
        self.prepare(n_points, seed).solve()
        for _ in range(n_epochs):
            self.epoch()

    @stages
    def prepare(self, n_points, seed):
        ran.append("prepare")
        self.xy = np.random.default_rng(seed).normal(size=(n_points, 2))

    @stages
    def solve(self):
        ran.append("solve")
        self.centre = self.load("xy").mean(axis=0)  # earlier results come from fields

    @stages
    def epoch(self):
        step = len(self.load("loss")) if "loss" in self else 0
        if step == crash["at"]:
            crash["at"] = None
            raise RuntimeError(f"node lost in epoch {step}")
        ran.append(f"epoch {step}")
        self.append("loss", [1.0 / (step + 1)])

    @stages
    def preview(self):
        self.summary = {"centre": self.load("centre").tolist()}


try:
    Fit()
except td.BuildFailed as exc:
    print("failed:", exc.error["message"], "| ran:", ran)

ran.clear()
fit = Fit.get(retry_failed=True)
print("resumed, ran:", ran, "| loss:", fit.load("loss").round(2).tolist())
{name: progress["calls"] for name, progress in fit.meta["stages"].items()}

The optional stage runs on the finished record; required stages refuse to.

In [ ]:
fit.preview()
print(fit.summary, fit.status)
try:
    fit.prepare(10, 0)
except td.ReadOnlyError as exc:
    print("ReadOnlyError:", exc)